# Implement e-mail spam filtering using text classification algorithm with appropriate dataset.
**Computer Laboratory II (417526) – Information Retrieval – Practical 4**

Install once (Linux terminal):
```
pip install nltk scikit-learn pandas matplotlib
```

In [ ]:
import os, re, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
warnings.filterwarnings("ignore")

DATA_DIR = "data"
def get_file(name, url):
    # Return path to data/<name>; download it first if it is not there yet.
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print("Downloading", url)
        urllib.request.urlretrieve(url, path)
    return path

In [ ]:
import nltk
# NLTK resources (needs internet once). The notebook still works without them using the fallbacks below.
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print("could not download", pkg, "->", e)

In [ ]:
from nltk.corpus import stopwords

# Offline copy of NLTK's English stop-word list (used only if the NLTK data cannot be downloaded)
NLTK_STOP_FALLBACK = set('''i me my myself we our ours ourselves you you're you've you'll you'd your yours yourself
yourselves he him his himself she she's her hers herself it it's its itself they them their theirs themselves what which
who whom this that that'll these those am is are was were be been being have has had having do does did doing a an the
and but if or because as until while of at by for with about against between into through during before after above below
to from up down in out on off over under again further then once here there when where why how all any both each few more
most other some such no nor not only own same so than too very s t can will just don don't should should've now d ll m o re
ve y ain aren aren't couldn couldn't didn didn't doesn doesn't hadn hadn't hasn hasn't haven haven't isn isn't ma mightn
mightn't mustn mustn't needn needn't shan shan't shouldn shouldn't wasn wasn't weren weren't won won't wouldn wouldn't'''.split())

text = '''Natural Language Processing (NLP) is a field of Artificial Intelligence. It helps computers read, process
and understand human language! Dr. Smith studies how machines are learning to translate languages better.
Search engines apply stemming for retrieving documents that match the queries of users.'''

# --- the stop-word list (NLTK, with an offline fallback) ---
try:
    STOP = set(stopwords.words("english"))
except LookupError:
    STOP = set(NLTK_STOP_FALLBACK)
    print("NLTK stopwords not available -> using the built-in copy of NLTK's English stop-word list")

In [ ]:
# ---- DATASET (embedded in this notebook) ----
# The data file is stored inside this notebook (compressed), so no download or internet is needed.
# Running this cell writes it to  data/sms.tsv
import base64, gzip, os
os.makedirs("data", exist_ok=True)
_DATA = """
...
"""
with open("data/sms.tsv", "wb") as f:
    f.write(gzip.decompress(base64.b64decode(_DATA.replace("\n", ""))))
print("data/sms.tsv written:", os.path.getsize("data/sms.tsv"), "bytes")

## Practical 4 – E-mail (SMS) spam filtering with k-NN text classification
**Pipeline:** collect data → pre-process (tokenize, lower-case, stop words, **Porter stemming**) → feature extraction (TF-IDF vectors) → **k-NN** classification (cosine distance; the manual shows Euclidean – cosine is the usual choice for text) → evaluation.

Evaluation uses the confusion matrix:
* Accuracy = (TP+TN)/(TP+FP+FN+TN)  * Precision = TP/(TP+FP)  * Sensitivity (recall) = TP/(TP+FN)  * Specificity = TN/(TN+FP)

Data set: SMS Spam Collection (5,572 messages labelled *ham* / *spam*), the `labels | message` format shown in the manual.

In [ ]:
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

sms = pd.read_csv(get_file("sms.tsv", "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"),
                  sep="\t", header=None, names=["label", "message"])
print(sms.shape); print(sms["label"].value_counts()); print(sms.head())

# ---- Step 2: pre-processing ----
stemmer = PorterStemmer()
def clean(msg):
    toks = re.findall(r"[a-z]+", msg.lower())
    return " ".join(stemmer.stem(t) for t in toks if t not in STOP)

sms["clean"] = sms["message"].apply(clean)
sms["y"] = (sms["label"] == "spam").astype(int)
print("\nBefore:", sms["message"][2][:80]); print("After :", sms["clean"][2][:80])

In [ ]:
# ---- Steps 3-4: features + train/test split ----
X_train, X_test, y_train, y_test = train_test_split(sms["clean"], sms["y"], test_size=0.2,
                                                    random_state=42, stratify=sms["y"])
vec = TfidfVectorizer(max_features=3000)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)
print("Feature matrix:", Xtr.shape)

# Tune k (hit-and-trial, as suggested in the manual)
ks = [1, 3, 5, 7, 9, 11, 15]
accs = []
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", algorithm="brute").fit(Xtr, y_train)
    accs.append(accuracy_score(y_test, knn.predict(Xte)))
    print(f"k={k:2d}  accuracy={accs[-1]:.4f}")
plt.plot(ks, accs, "o-"); plt.xlabel("k"); plt.ylabel("test accuracy"); plt.grid(alpha=.3); plt.show()
best_k = ks[int(np.argmax(accs))]
print("Best k =", best_k)

In [ ]:
# ---- Step 5: performance analysis ----
knn = KNeighborsClassifier(n_neighbors=best_k, metric="cosine", algorithm="brute").fit(Xtr, y_train)
pred = knn.predict(Xte)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print("Confusion matrix (rows = actual ham/spam, columns = predicted ham/spam)")
print(pd.DataFrame([[tn, fp], [fn, tp]], index=["ham", "spam"], columns=["pred ham", "pred spam"]))
print(f"\nAccuracy    : {(tp+tn)/(tp+tn+fp+fn):.4f}")
print(f"Precision   : {tp/(tp+fp):.4f}")
print(f"Sensitivity : {tp/(tp+fn):.4f}")
print(f"Specificity : {tn/(tn+fp):.4f}")

# classify new messages
new = ["WINNER!! You have won a free prize. Call now to claim your cash reward",
       "Hey, are we still meeting for lunch tomorrow?"]
for m, p in zip(new, knn.predict(vec.transform([clean(x) for x in new]))):
    print("SPAM" if p else "HAM ", "|", m)